# 10. Data Integration

Checking empirical referential integrity, cardinality, and creating the final denormalized table.

In [1]:
import pandas as pd

customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

## Referential Integrity Checks

In [2]:
print('Orphaned order_items:', items[~items['order_id'].isin(orders['order_id'])].shape[0])
print('Orphaned order_payments:', payments[~payments['order_id'].isin(orders['order_id'])].shape[0])
print('Orphaned order_reviews:', reviews[~reviews['order_id'].isin(orders['order_id'])].shape[0])

Orphaned order_items: 0
Orphaned order_payments: 0
Orphaned order_reviews: 0


In [3]:
print('Orders missing a customer:', orders[~orders['customer_id'].isin(customers['customer_id'])].shape[0])

Orders missing a customer: 0


In [4]:
print('Items missing a product:', items[~items['product_id'].isin(products['product_id'])].shape[0])

Items missing a product: 0


In [5]:
print('Items missing a seller:', items[~items['seller_id'].isin(sellers['seller_id'])].shape[0])

Items missing a seller: 0


## Empirical Cardinality Checks

In [6]:
print('Items per order stats:')
print(items.groupby('order_id').size().describe())
print('\nPayments per order stats:')
print(payments.groupby('order_id').size().describe())
print('\nReviews per order stats:')
print(reviews.groupby('order_id').size().describe())

Items per order stats:
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

Payments per order stats:
count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

Reviews per order stats:
count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64


## Building the Master Table
We denormalize at the `order_items` grain, pre-aggregating payments and reviews first.

In [7]:
# Pre-aggregate payments: sum the payment value per order
payments_agg = payments.groupby('order_id', as_index=False)['payment_value'].sum()

# Pre-aggregate reviews: take the mean review score per order
reviews_agg = reviews.groupby('order_id', as_index=False)['review_score'].mean()

# Merge translations to products
products_en = pd.merge(products, translation, on='product_category_name', how='left')

# Master Merge
# Base is order_items (grain of the final table)
master = items.merge(orders, on='order_id', how='left')
master = master.merge(customers, on='customer_id', how='left')
master = master.merge(sellers, on='seller_id', how='left')
master = master.merge(products_en, on='product_id', how='left')
master = master.merge(payments_agg, on='order_id', how='left')
master = master.merge(reviews_agg, on='order_id', how='left')


In [8]:
print('Original order_items rows:', len(items))
print('Master table rows:', len(master))
print('Are row counts equal?', len(items) == len(master))

Original order_items rows: 112650
Master table rows: 112650
Are row counts equal? True


In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)
master.to_csv('../data/processed/master_table.csv', index=False)
print('Saved to data/processed/master_table.csv')

Saved to data/processed/master_table.csv


### Takeaways
- Referential integrity holds perfectly across all tables (0 orphaned rows found).
- Pre-aggregating payments (total sum) and reviews (average score) successfully preserved the `order_items` grain, avoiding row duplication (fan-out) upon merge.
- The final dataset was merged cleanly and exported, maintaining the expected 112,650 item-level rows.